# 02 — Phase 4.5 GPU runner
Validation-only modular runner. It does not start until the control cell is executed.

## 1. Runtime check

In [ ]:
import os, pathlib, platform, shutil, subprocess, time, torch
REQUIRE_GPU=True
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('CUDA:',torch.version.cuda,'VRAM GiB:',torch.cuda.get_device_properties(0).total_memory/2**30 if torch.cuda.is_available() else 0)
print('Python:',platform.python_version(),'PyTorch:',torch.__version__)
if REQUIRE_GPU and not torch.cuda.is_available(): raise RuntimeError('Enable a GPU runtime')


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT='/content/drive/MyDrive/evo2_dissertation'
assert pathlib.Path(DRIVE_ROOT).is_dir()


## 3. Acquire GitHub code

In [ ]:
REPO_URL='REPLACE_WITH_GITHUB_REPO_URL'
REPO_ROOT='/content/evo2-dissertation'
assert 'REPLACE_' not in REPO_URL, 'Set REPO_URL'
if pathlib.Path(REPO_ROOT,'.git').is_dir(): subprocess.run(['git','-C',REPO_ROOT,'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone',REPO_URL,REPO_ROOT],check=True)


## 4. Install environment

In [ ]:
subprocess.run(['pip','install','-q','-r',f'{REPO_ROOT}/requirements/requirements-colab.txt'],check=True)
subprocess.run(['pip','install','-q','-e',REPO_ROOT],check=True)
subprocess.run(['python',f'{REPO_ROOT}/scripts/verify_environment.py','--require-gpu'],check=True)


## 5. Verify manifests

In [ ]:
subprocess.run(['python',f'{REPO_ROOT}/scripts/verify_data_manifest.py','--drive-root',DRIVE_ROOT],check=True)


## 6. Stage data to local SSD

In [ ]:
LOCAL_DATA='/content/evo2_dissertation_data'
LOCAL_RUNS='/content/evo2_runs'
stage=['python',f'{REPO_ROOT}/scripts/stage_drive_data.py','--drive-root',DRIVE_ROOT,'--local-root',LOCAL_DATA]
if not pathlib.Path(DRIVE_ROOT,'caches/tokens/token_cache_manifest.json').exists(): stage.append('--include-fasta')
subprocess.run(stage,check=True)


## 7. Load/build token cache

In [ ]:
cache=pathlib.Path(LOCAL_DATA,'caches/tokens')
if (cache/'token_cache_manifest.json').exists():
    subprocess.run(['python',f'{REPO_ROOT}/scripts/build_token_cache.py','--drive-root',LOCAL_DATA,'--output-dir',str(cache),'--verify-only'],check=True)
else:
    subprocess.run(['python',f'{REPO_ROOT}/scripts/build_token_cache.py','--drive-root',LOCAL_DATA,'--output-dir',str(cache)],check=True)


## 8. GPU benchmark — infrastructure only

In [ ]:
BENCHMARK='/content/gpu_benchmark.json'
subprocess.run(['python',f'{REPO_ROOT}/scripts/benchmark_gpu.py','--output',BENCHMARK,'--device','cuda'],check=True)
from evo2_distill.utils.io import atomic_copy
atomic_copy(BENCHMARK,pathlib.Path(DRIVE_ROOT,'results/phase4_5/gpu_benchmark.json'))


## 9. Phase 4.5 execution controls

In [ ]:
RUN_DATA_SCALING=True
RUN_CAPACITY_SCALING=True
RUN_TAIL_AWARE=True
RUN_BASELINE_CORRECTION=True
RUN_FINAL_ENSEMBLE=False
RUN_UQ_ANALYSIS=False
os.environ['EVO2_DATA_ROOT']=LOCAL_DATA
os.environ['EVO2_LOCAL_RUN_ROOT']=LOCAL_RUNS


## 10. Run scientific experiment modules
Only selected booleans run. TEST is rejected by code-level gates.

In [ ]:
controls={'data_scaling':RUN_DATA_SCALING,'capacity_scaling':RUN_CAPACITY_SCALING,'tail_aware':RUN_TAIL_AWARE,'baseline_correction':RUN_BASELINE_CORRECTION,'final_ensemble':RUN_FINAL_ENSEMBLE}
completed=[]; failed=[]
for name,enabled in controls.items():
    if not enabled: continue
    command=['python',f'{REPO_ROOT}/scripts/run_phase4_5.py','--config',f'{REPO_ROOT}/configs/phase4_5/{name}.yaml']
    result=subprocess.run(command)
    (completed if result.returncode==0 else failed).append(name)
if RUN_UQ_ANALYSIS: print('Run notebook 04 after candidate outputs have been synced.')


## 11. Persist results

In [ ]:
for run in pathlib.Path(LOCAL_RUNS).glob('*') if pathlib.Path(LOCAL_RUNS).exists() else []:
    if run.is_dir(): subprocess.run(['python',f'{REPO_ROOT}/scripts/sync_results_to_drive.py','--local-run',str(run),'--drive-results-root',f'{DRIVE_ROOT}/results/phase4_5'],check=True)


## 12. Final summary

In [ ]:
git_commit=subprocess.check_output(['git','-C',REPO_ROOT,'rev-parse','HEAD'],text=True).strip()
print('completed experiments:',completed)
print('failed experiments:',failed)
print('resumable experiments:',[p.name for p in pathlib.Path(LOCAL_RUNS).glob('*') if (p/'checkpoint').is_dir()])
print('Drive result path:',f'{DRIVE_ROOT}/results/phase4_5')
print('Git commit hash:',git_commit)
print('config hashes: recorded in each run_manifest.json')
print('TEST ACCESSED: NO')
print('TEST REMAINS LOCKED: YES')
